# Spectral Clustering — clustering the graph, not the coordinates

> Tutorial pair for [`spectral.py`](spectral.py).

## 1. Intuition
Two points on opposite ends of a curved "moon" are far apart in space but
*connected* through a chain of near neighbors. Spectral clustering turns the data
into a **similarity graph**, then uses the eigenvectors of its **Laplacian** to
find a low-dimensional embedding where those connected points sit together.
Ordinary k-means in that embedding then separates shapes that k-means could never
separate in the original space.

## 2. Concept (the slide)
1. **Affinity** $W$: edge weight $w_{ij}=\exp(-\gamma\lVert x_i-x_j\rVert^2)$
   (optionally sparsified to a $k$-NN graph).
2. **Laplacian** $L=D-W$ with degree $D=\mathrm{diag}(\sum_j w_{ij})$ — or a
   normalized variant.
3. **Embed**: take the eigenvectors of the $k$ *smallest* eigenvalues $\to$
   $U\in\mathbb R^{n\times k}$.
4. **k-means** on the rows of $U$ gives the clusters.

The smallest eigenvectors are the *smoothest* functions on the graph — nearly
constant within well-connected components — which is exactly what makes the
clusters fall out.

## 3. Math derivation

**The graph Laplacian.** For affinity $W\succeq 0$ and degrees $d_i=\sum_j w_{ij}$,
$L=D-W$. For any vector $f\in\mathbb R^n$,
$$f^\top L f=\tfrac12\sum_{i,j} w_{ij}\,(f_i-f_j)^2\ \ge 0,$$
so $L$ is positive semidefinite. The all-ones vector gives $L\mathbf 1=0$, so
$\lambda_1=0$; the multiplicity of eigenvalue $0$ equals the number of connected
components, and those eigenvectors are constant on each component — the ideal
cluster indicators.

**Graph cut $\to$ eigenproblem.** Partition $V$ into $A,\bar A$. The cut is
$\mathrm{cut}(A,\bar A)=\sum_{i\in A,j\in\bar A} w_{ij}$. Minimizing it alone
peels off single vertices, so we *balance* it — the **normalized cut**
$$\mathrm{Ncut}(A,\bar A)=\mathrm{cut}(A,\bar A)\Big(\tfrac{1}{\mathrm{vol}(A)}+\tfrac{1}{\mathrm{vol}(\bar A)}\Big),
\quad \mathrm{vol}(A)=\sum_{i\in A} d_i.$$
With a $\{+,-\}$ indicator $f$ encoding the partition, one shows
$\mathrm{Ncut}\propto \dfrac{f^\top L f}{f^\top D f}$ subject to $f\perp_D \mathbf 1$.
Exact minimization over discrete $f$ is **NP-hard**; **relax** $f$ to real values:
$$\min_{f}\ \frac{f^\top L f}{f^\top D f}\quad\text{s.t. } f^\top D\mathbf 1=0.$$
This Rayleigh quotient is minimized by the **second-smallest generalized
eigenvector** of $L f=\lambda D f$ — the *Fiedler vector*. For $k>2$ clusters,
take the $k$ smallest eigenvectors and cluster their rows.

**Normalized Laplacians.** Solving $Lf=\lambda Df$ corresponds to
- random-walk: $L_{rw}=I-D^{-1}W$ (Shi-Malik), eigenvectors of $D^{-1}W$;
- symmetric: $L_{sym}=I-D^{-1/2}WD^{-1/2}=D^{1/2}L_{rw}D^{-1/2}$
  (Ng-Jordan-Weiss). For $L_{sym}$ the embedding rows are **renormalized to unit
  length** before k-means.

**Laplacian eigenmaps view.** The embedding $x_i\mapsto U_{i,:}$ minimizes
$\sum_{ij} w_{ij}\lVert U_{i,:}-U_{j,:}\rVert^2 = \mathrm{tr}(U^\top L U)$ subject
to orthonormality — strongly connected points are pulled together, so the
embedding makes clusters linearly separable for k-means.

**Eigengap heuristic.** Sort the eigenvalues $0=\lambda_1\le\lambda_2\le\cdots$;
a large gap $\lambda_{k+1}-\lambda_k$ suggests $k$ clusters.

## 4. NumPy implementation (RBF/kNN affinity, 3 Laplacians, eigen-embed + k-means)

In [ ]:
# ===== actual implementation from spectral.py =====
from __future__ import annotations

import numpy as np

SEED = 0

def _kmeans(X, k, n_iters=100, n_init=10, seed=SEED):
    """Small self-contained k-means++ for the embedding step (no sibling import)."""
    X = np.asarray(X, float)
    rng = np.random.default_rng(seed)
    best_inertia, best_labels = np.inf, None
    for _ in range(n_init):
        C = [X[rng.integers(len(X))]]
        for _ in range(1, k):
            d2 = ((X[:, None, :] - np.array(C)[None, :, :]) ** 2).sum(2).min(1)
            C.append(X[rng.choice(len(X), p=d2 / d2.sum())])
        C = np.array(C, float)
        for _ in range(n_iters):
            D = ((X[:, None, :] - C[None, :, :]) ** 2).sum(2)
            lab = D.argmin(1)
            newC = np.array([X[lab == j].mean(0) if np.any(lab == j) else C[j]
                             for j in range(k)])
            if np.allclose(newC, C):
                C = newC; break
            C = newC
        inertia = ((X - C[lab]) ** 2).sum()
        if inertia < best_inertia:
            best_inertia, best_labels = inertia, lab
    return best_labels

import torch

def get_device():
    """Pick the best available device: cuda > mps > cpu."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    from sklearn.datasets import make_moons, make_circles, make_blobs
    from sklearn.metrics import adjusted_rand_score

    # two moons: spectral separates the non-convex shapes; k-means cannot
    Xm, ym = make_moons(n_samples=300, noise=0.06, random_state=SEED)
    sc = SpectralClusteringNumPy(2, affinity="rbf", gamma=15, laplacian="sym").fit(Xm)
    km = _kmeans(Xm, 2)
    print(f"Moons:   spectral ARI={adjusted_rand_score(ym, sc.labels_):.3f}  "
          f"k-means ARI={adjusted_rand_score(ym, km):.3f}")

    # concentric circles: same story
    Xc, yc = make_circles(n_samples=300, factor=0.4, noise=0.05, random_state=SEED)
    scc = SpectralClusteringNumPy(2, affinity="rbf", gamma=15, laplacian="sym").fit(Xc)
    print(f"Circles: spectral ARI={adjusted_rand_score(yc, scc.labels_):.3f}  "
          f"k-means ARI={adjusted_rand_score(yc, _kmeans(Xc, 2)):.3f}")

    print("\nLaplacian variants on moons (ARI):")
    for lap in ("unnormalized", "sym", "rw"):
        s = SpectralClusteringNumPy(2, gamma=15, laplacian=lap).fit(Xm)
        print(f"  {lap:12s}: {adjusted_rand_score(ym, s.labels_):.3f}")

    lab, U = spectral_torch(Xm, n_clusters=2, gamma=15)
    print(f"\nTorch spectral (sym) ARI={adjusted_rand_score(ym, lab):.3f}  device={get_device()}")

    # the "eigengap" hints at the number of clusters
    Xb, yb = make_blobs(n_samples=300, centers=4, cluster_std=0.6, random_state=SEED)
    sb = SpectralClusteringNumPy(4, gamma=2, laplacian="sym").fit(Xb)
    print(f"\nBlobs(k=4) smallest eigenvalues: {np.round(sb.eigenvalues_, 3)} "
          f"(gap after #clusters)")


class SpectralClusteringNumPy:
    r"""
    Graph-cut view. Partition the similarity graph to minimize the **normalized
    cut**. With cluster indicator vectors this is NP-hard; relaxing the indicators
    to real values turns it into a generalized eigenproblem on the Laplacian.

    Laplacians (W = affinity, D = degree diag):
      unnormalized: L      = D - W
      symmetric:    L_sym  = I - D^{-1/2} W D^{-1/2}
      random walk:  L_rw   = I - D^{-1} W   (eigvecs = those of  D^{-1}W)

    The k eigenvectors with the smallest eigenvalues give a piecewise-constant
    embedding on which ordinary k-means recovers the clusters. For L_sym the
    rows are renormalized to unit length (Ng-Jordan-Weiss).
    """

    def __init__(self, n_clusters=2, affinity="rbf", gamma=1.0, n_neighbors=10,
                 laplacian="sym", seed=SEED):
        assert laplacian in ("unnormalized", "sym", "rw")
        self.n_clusters, self.affinity, self.gamma = n_clusters, affinity, gamma
        self.n_neighbors, self.laplacian, self.seed = n_neighbors, laplacian, seed

    def _affinity(self, X):
        d2 = ((X[:, None, :] - X[None, :, :]) ** 2).sum(2)
        if self.affinity == "knn":
            # symmetric k-NN graph: connect i~j if either is in the other's kNN
            W = np.zeros_like(d2)
            idx = np.argsort(d2, axis=1)[:, 1:self.n_neighbors + 1]
            for i in range(len(X)):
                W[i, idx[i]] = np.exp(-self.gamma * d2[i, idx[i]])
            W = np.maximum(W, W.T)                     # make symmetric
        else:                                         # full RBF (Gaussian) affinity
            W = np.exp(-self.gamma * d2)
            np.fill_diagonal(W, 0.0)                   # no self-loops
        return W

    def fit(self, X):
        X = np.asarray(X, float)
        W = self._affinity(X)
        deg = W.sum(1)                                 # node degrees
        D = np.diag(deg)

        if self.laplacian == "unnormalized":
            L = D - W
            vals, vecs = np.linalg.eigh(L)
            U = vecs[:, :self.n_clusters]              # smallest eigenvalues
        elif self.laplacian == "sym":
            d_inv_sqrt = 1.0 / np.sqrt(deg + 1e-12)
            Lsym = np.eye(len(X)) - (d_inv_sqrt[:, None] * W * d_inv_sqrt[None, :])
            vals, vecs = np.linalg.eigh(Lsym)
            U = vecs[:, :self.n_clusters]
            U = U / (np.linalg.norm(U, axis=1, keepdims=True) + 1e-12)  # row-normalize
        else:  # random walk: solve L_rw u = lambda u  <=>  generalized (D-W)u=lambda D u
            d_inv = 1.0 / (deg + 1e-12)
            Lrw = np.eye(len(X)) - d_inv[:, None] * W
            vals, vecs = np.linalg.eig(Lrw)            # Lrw is not symmetric
            order = np.argsort(vals.real)
            U = vecs[:, order[:self.n_clusters]].real

        self.embedding_ = U
        self.eigenvalues_ = np.sort(vals.real)[:self.n_clusters + 1]
        self.affinity_matrix_ = W
        self.labels_ = _kmeans(U, self.n_clusters, seed=self.seed)
        return self

    def fit_predict(self, X):
        return self.fit(X).labels_

## 5. PyTorch implementation (torch.cdist affinity + torch.linalg.eigh)

In [ ]:
# ===== actual implementation from spectral.py =====
def spectral_torch(X, n_clusters=2, gamma=1.0, seed=SEED):
    """Symmetric-normalized spectral embedding via torch, then NumPy k-means.

    Heavy linear algebra (affinity via torch.cdist, eigh of the Laplacian) runs
    on the device; only the tiny final k-means runs on CPU/NumPy.
    """
    dev = get_device()
    Xt = torch.as_tensor(np.asarray(X, np.float32), device=dev)
    D2 = torch.cdist(Xt, Xt) ** 2                      # squared distances
    W = torch.exp(-gamma * D2)
    W.fill_diagonal_(0.0)
    deg = W.sum(1)
    d_inv_sqrt = (deg + 1e-12).rsqrt()
    Lsym = torch.eye(len(Xt), device=dev) - d_inv_sqrt[:, None] * W * d_inv_sqrt[None, :]
    vals, vecs = torch.linalg.eigh(Lsym)               # ascending eigenvalues
    U = vecs[:, :n_clusters]
    U = U / (U.norm(dim=1, keepdim=True) + 1e-12)
    labels = _kmeans(U.cpu().numpy(), n_clusters, seed=seed)
    return labels, U.cpu().numpy()

## 6. Train / run — moons & circles vs k-means, Laplacian variants, eigengap

In [ ]:
demo()

## 7. Visualization — input clusters and the spectral embedding

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_moons
import spectral as M

X, y = make_moons(n_samples=300, noise=0.06, random_state=0)
sc = M.SpectralClusteringNumPy(2, affinity="rbf", gamma=15, laplacian="sym").fit(X)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(X[:, 0], X[:, 1], c=sc.labels_, s=14, cmap="coolwarm")
ax[0].set_title("Spectral clustering (two moons)")
# the 2-D embedding: clusters become two tight, linearly separable blobs
U = sc.embedding_
ax[1].scatter(U[:, 0], U[:, 1], c=sc.labels_, s=14, cmap="coolwarm")
ax[1].set_xlabel("eigvec 1"); ax[1].set_ylabel("eigvec 2")
ax[1].set_title("Laplacian eigen-embedding (k-means lives here)")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- **Handles non-convex clusters** (moons, rings) by clustering graph connectivity
  rather than Euclidean proximity — where plain k-means fails.
- **Affinity scale $\gamma$ is critical**: too large disconnects the graph, too
  small connects everything. The $k$-NN graph is often more robust than full RBF.
- **Normalized** Laplacians ($L_{sym}$, $L_{rw}$) usually beat the unnormalized
  one when degrees vary; remember to **row-normalize** the $L_{sym}$ embedding.
- Cost is $O(n^2)$ memory and an $O(n^3)$ dense eigensolve — use sparse $k$-NN
  graphs + Lanczos for large $n$. Read $k$ from the **eigengap**.

**Next:** leave clustering for *dimensionality reduction* — find the directions
of maximum variance with PCA.